# Модуль 2 · Лекція 1 — Умовні оператори та цикли

**Передумова:** Модуль 1 (типи даних, колекції, truthiness)

## Що ви знатимете після лекції
- умовні конструкції: `if/elif/else`, тернарний оператор, `match/case` (3.10+);
- цикли `for` і `while`, `range`, `enumerate`, `zip`;
- `break`, `continue` та рідкісний, але цінний `for...else`;
- **протокол ітератора** (`__iter__`, `__next__`, `StopIteration`) — що таке iterable vs iterator;
- **генератори** (`yield`) та генераторні вирази — лінива обробка даних;
- модуль `itertools`.

Наприкінці — **⚡ каверзні питання** про цикли й генератори.

## 1. Умовні конструкції

`if/elif/else` виконує різні гілки залежно від умови. Блоки визначаються **відступами** (4 пробіли). Умова оцінюється за правилами truthiness (див. Модуль 1).

In [ ]:
score = 82

if score >= 90:
    grade = "A"
elif score >= 75:
    grade = "B"
elif score >= 60:
    grade = "C"
else:
    grade = "F"

print("Оцінка:", grade)

In [ ]:
# Комбінуємо умови логічними операторами (and / or / not)
age = 25
has_ticket = True

if age >= 18 and has_ticket:
    print("Пропускаємо")
elif age >= 18 and not has_ticket:
    print("Потрібен квиток")
else:
    print("Вік не дозволяє")

# Вкладені if — коли рішення багатоступеневе
temp = 30
if temp > 0:
    if temp > 25:
        print("Спекотно")
    else:
        print("Тепло")
else:
    print("Мороз")

In [ ]:
# in та ланцюгове порівняння прямо в умові
day = "Sat"
if day in ("Sat", "Sun"):
    print("Вихідний")

x = 50
if 0 <= x <= 100:              # еквівалент (0 <= x) and (x <= 100)
    print("x у діапазоні [0, 100]")

# elif перевіряється ПО ПОРЯДКУ й зупиняється на першій істинній умові
n = 0
if n > 0:
    print("додатне")
elif n < 0:
    print("відʼємне")
else:
    print("нуль")

### 1.1. Тернарний оператор (умовний вираз)
Компактний `if/else` в один рядок: `X if умова else Y`.

In [ ]:
age = 20
status = "дорослий" if age >= 18 else "неповнолітній"
print(status)

# Часто для значень за замовчуванням:
name = ""
display = name if name else "Гість"
print(display)

### 1.2. `match/case` — структурне зіставлення (Python 3.10+)
Аналог `switch`, але потужніший: вміє зіставляти **структуру** даних, а не лише значення.

In [ ]:
def http_status(code):
    match code:
        case 200:
            return "OK"
        case 400 | 401 | 403:        # кілька варіантів
            return "Помилка клієнта"
        case 500:
            return "Помилка сервера"
        case _:                       # _ — "все інше" (default)
            return "Невідомо"

for code in [200, 401, 500, 302]:
    print(code, "->", http_status(code))

In [ ]:
# match вміє РОЗПАКОВУВАТИ структури — це його головна сила
def describe(point):
    match point:
        case (0, 0):
            return "початок координат"
        case (x, 0):
            return f"на осі X, x={x}"
        case (0, y):
            return f"на осі Y, y={y}"
        case (x, y):
            return f"точка ({x}, {y})"

print(describe((0, 0)))
print(describe((5, 0)))
print(describe((3, 4)))

## 2. Цикли `for` і `while`

- `for` — перебирає елементи будь-якого **ітерабельного** об'єкта (список, рядок, dict, файл...).
- `while` — повторює, поки умова істинна.

In [ ]:
# range(start, stop, step) — stop не включається
for i in range(1, 6):
    print(i, end=" ")
print()

# while з умовою виходу
n = 5
factorial = 1
while n > 1:
    factorial *= n
    n -= 1
print("5! =", factorial)

### 2.1. `enumerate` та `zip` — must-have інструменти

- `enumerate(iterable)` — дає пари `(індекс, елемент)`. **Не пишіть `range(len(...))`!**
- `zip(a, b)` — «зшиває» кілька послідовностей у пари.

In [ ]:
fruits = ["яблуко", "банан", "вишня"]

# Правильно — enumerate замість range(len(...))
for index, fruit in enumerate(fruits, start=1):
    print(f"{index}. {fruit}")

# zip — паралельний перебір
names = ["Аня", "Богдан"]
ages = [25, 30]
for name, age in zip(names, ages):
    print(f"{name} — {age} р.")

# zip зупиняється по найкоротшій послідовності
print(list(zip([1, 2, 3], ["a", "b"])))   # [(1,'a'), (2,'b')]

### 2.2. `break`, `continue` та `for...else`

- `break` — негайно вийти з циклу.
- `continue` — пропустити решту тіла й перейти до наступної ітерації.
- **`for...else`**: блок `else` виконується, **якщо цикл завершився без `break`**. Малознане, але корисне.

In [ ]:
# continue: пропускаємо парні
for n in range(1, 8):
    if n % 2 == 0:
        continue
    print("непарне:", n)

In [ ]:
# for...else — типовий сценарій "пошук": else = "не знайшли"
numbers = [4, 6, 8, 10]

for n in numbers:
    if n % 2 != 0:
        print("Знайдено непарне:", n)
        break
else:
    # виконається ТІЛЬКИ якщо break не спрацював
    print("Непарних чисел немає — усі парні")

## 3. Ітератори та ітерабельні об'єкти ⭐

Це фундамент, на якому побудований `for`. Два різні поняття:

- **Iterable (ітерабельний)** — об'єкт, який можна перебирати; має метод `__iter__()`, що повертає ітератор. Приклади: `list`, `str`, `dict`, `range`.
- **Iterator (ітератор)** — об'єкт, який власне видає елементи по одному через `__next__()`; коли елементи закінчуються — кидає `StopIteration`. Ітератор також має `__iter__` (повертає себе).

**Цикл `for` під капотом:** викликає `iter(obj)` → отримує ітератор → у циклі кличе `next()`, поки не впіймає `StopIteration`.

In [ ]:
nums = [10, 20, 30]

it = iter(nums)          # отримали ітератор зі списку (iterable)
print(type(it))
print(next(it))          # 10
print(next(it))          # 20
print(next(it))          # 30
try:
    next(it)             # елементи скінчились
except StopIteration:
    print("StopIteration — елементи закінчились")

In [ ]:
# Отак приблизно "for" працює зсередини:
def my_for(iterable, action):
    iterator = iter(iterable)
    while True:
        try:
            item = next(iterator)
        except StopIteration:
            break
        action(item)

my_for(["a", "b", "c"], lambda x: print("елемент:", x))

In [ ]:
# Власний ітератор: клас, що рахує від 1 до n
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self                      # сам є своїм ітератором

    def __next__(self):
        if self.current <= 0:
            raise StopIteration          # сигнал завершення
        self.current -= 1
        return self.current + 1

for x in Countdown(3):
    print("countdown:", x)

## 4. Генератори (`yield`) ⭐

Генератор — **найпростіший спосіб створити ітератор**. Це функція, яка замість `return` використовує **`yield`**. Виклик такої функції не виконує тіло одразу, а повертає **генератор-об'єкт** (ітератор).

Ключова ідея — **лінива (lazy) обробка**: значення обчислюються по одному, **лише коли їх запитують**. Тому генератор:
- майже не займає пам'ять (не тримає всі елементи одразу);
- може представляти навіть **нескінченну** послідовність;
- **вичерпується** — по ньому можна пройти лише один раз.

In [ ]:
def countdown(n):
    print("  генератор стартував")
    while n > 0:
        yield n           # "віддає" значення і ЗАСТИГАЄ тут до наступного next()
        n -= 1

gen = countdown(3)        # тіло ще НЕ виконувалось!
print("створили генератор:", gen)
print(next(gen))          # аж тепер друкує "стартував" і повертає 3
print(next(gen))          # 2
print(next(gen))          # 1

In [ ]:
# Пам'ять: порівняємо список і генератор для великого діапазону
import sys

list_version = [x * x for x in range(100_000)]     # усе в пам'яті
gen_version = (x * x for x in range(100_000))       # генераторний вираз — лінивий

print("список займає байтів:   ", sys.getsizeof(list_version))
print("генератор займає байтів:", sys.getsizeof(gen_version))  # у сотні разів менше

In [ ]:
# Вичерпність: генератор проходиться ТІЛЬКИ раз
gen = (x for x in range(3))
print("перший прохід:", list(gen))   # [0, 1, 2]
print("другий прохід:", list(gen))   # [] — вже порожній!

In [ ]:
# Нескінченний генератор + зупинка ззовні
def naturals():
    n = 1
    while True:          # нескінченно, але це ОК завдяки лінивості
        yield n
        n += 1

gen = naturals()
first_five = [next(gen) for _ in range(5)]
print(first_five)        # [1, 2, 3, 4, 5]

## 5. `itertools` — інструменти для ітерування

Стандартний модуль зі швидкими будівельними блоками: `count`, `cycle`, `chain`, `islice`, `combinations`, `groupby` тощо.

In [ ]:
import itertools

# islice — "зріз" для будь-якого (навіть нескінченного) ітератора
print(list(itertools.islice(itertools.count(10, 5), 4)))   # [10, 15, 20, 25]

# chain — послідовно об'єднати кілька послідовностей
print(list(itertools.chain([1, 2], [3, 4], [5])))

# combinations — усі пари
print(list(itertools.combinations(["A", "B", "C"], 2)))

## ⚡ Каверзні питання

**Q1. Чим iterable відрізняється від iterator?**
*Iterable* можна перебирати — має `__iter__()`, що повертає *iterator*. *Iterator* видає елементи через `__next__()` і вичерпується. `list` — iterable, але не iterator; `iter(list)` дає iterator.

**Q2. У чому перевага генератора над списком?**
Лінива обробка: генератор не тримає всі елементи в пам'яті, обчислює їх на вимогу, може бути нескінченним. Мінус — по ньому можна пройти лише раз.

In [ ]:
# Q3. Що надрукує другий list(gen)?
gen = (x * 2 for x in range(3))
print(list(gen))   # [0, 2, 4]
print(list(gen))   # ?

**Q3 (відповідь).** Другий — `[]`. Генератор **вичерпався** після першого проходу; повторно він не «перезапускається».

In [ ]:
# Q4. Коли виконається блок else?
def find_first_negative(nums):
    for n in nums:
        if n < 0:
            return n
    else:
        return "немає від'ємних"

print(find_first_negative([1, 2, 3]))    # ?
print(find_first_negative([1, -2, 3]))   # ?

**Q4 (відповідь).** Перший — `"немає від'ємних"` (цикл дійшов до кінця без `return`/`break`, тож `else` спрацював). Другий — `-2` (вихід із функції стався до завершення циклу, `else` не виконується).

In [ ]:
# Q5. Чому це поганий код і як його виправити?
fruits = ["a", "b", "c"]
# погано:
for i in range(len(fruits)):
    print(i, fruits[i])
# добре:
for i, fruit in enumerate(fruits):
    print(i, fruit)

**Q5 (відповідь).** `range(len(...))` — неідіоматично й ламкіше. `enumerate` одразу дає індекс і елемент, читабельніше й безпечніше.

## 🎯 Практичні вправи

1. Напишіть `match/case`, який за днем тижня (`1..7`) повертає `"робочий"` або `"вихідний"`.
2. Використовуючи `enumerate`, виведіть список покупок у форматі `"1) хліб"`, `"2) молоко"`, ...
3. За допомогою `zip` створіть словник `{ім'я: вік}` з двох списків.
4. Напишіть цикл із `for...else`, що шукає перше число, яке ділиться на 7, у списку; якщо такого немає — повідомляє про це.
5. Напишіть **генератор** `even_numbers(limit)`, який ліниво видає парні числа до `limit`. Доведіть `sys.getsizeof`, що він компактніший за список.
6. Напишіть генератор Фібоначчі й візьміть з нього перші 10 чисел через `itertools.islice`.
7. (⭐) Продемонструйте кодом вичерпність генератора: пройдіться по ньому двічі й поясніть результат.

In [ ]:
# Місце для ваших вправ


## 📚 Додаткові матеріали
- Керування потоком, цикли (укр.): https://docs.python.org/uk/3/tutorial/controlflow.html
- `match/case` — PEP 636 (туторіал): https://peps.python.org/pep-0636/
- Ітератори та генератори (офіційно): https://docs.python.org/3/howto/functional.html
- `itertools`: https://docs.python.org/3/library/itertools.html
- Real Python — Generators: https://realpython.com/introduction-to-python-generators/
- Real Python — Iterators & Iterables: https://realpython.com/python-iterators-iterables/